# 🛍️ RETAIL RADAR
## From Transaction Rows to Business Decisions

**Project type:** Exploratory Data Analysis (EDA)  
**Tools:** Python · pandas · NumPy · Matplotlib · Seaborn · Jupyter Notebook

> **Project objective**  
> Discover sales trends, customer demographics, product performance, and less-obvious revenue patterns, then turn the evidence into practical recommendations.

### Questions this notebook answers
1. How do sales change month by month and quarter by quarter?
2. Which age groups and gender groups appear most often in the transaction data?
3. Which products sell the most units, and which categories generate the most revenue?
4. Which numerical variables move together, and how concentrated is revenue among top products?
5. What should the business test or improve next?

### Data-source note
This starter notebook is set up for the **`sales_data.csv` simulated retail dataset** from [this public GitHub Gist](https://gist.github.com/summerofgeorge/33dbb09adbdea670c31ee2c4fc9f2617). It contains product names, transaction dates, quantities, unit prices, age, and gender, so it can support every chart in the assignment. Because the source describes the data as simulated, treat the results as a demonstration of the analytical workflow—not as evidence about a real retailer. If your internship mentor specifically requires a Kaggle dataset, select one that includes both actual product names and customer age/gender plus dates; the commonly used 1,000-row Kaggle “Retail Sales Dataset” has demographics but only product categories, so its data alone cannot truthfully identify ten individual products.

**Required local file:** `data/sales_data.csv`  
Download it from the Gist link above by opening `sales_data.csv` → **Raw** → save it as `sales_data.csv` inside the `data` folder beside this notebook.

## 1. Import libraries and set a consistent visual style

The dark-teal, evergreen, and warm-amber palette is used throughout so the charts look like one professional analytics report. All chart observations later in the notebook are calculated from the loaded dataset; do not copy generic findings before you run the code.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)

# Project folders. This assumes Jupyter was launched from the project root.
DATA_PATH = Path("data/sales_data.csv")
OUTPUT_DIR = Path("outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# RETAIL RADAR visual identity
INK = "#17324D"
TEAL = "#178F8B"
GREEN = "#2D6A4F"
AMBER = "#F4A261"
CORAL = "#E76F51"
PALE = "#EAF2F4"

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)
sns.set_palette([TEAL, AMBER, GREEN, CORAL, INK])
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.titleweight": "bold",
    "axes.titlecolor": INK,
    "axes.labelcolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "font.family": "DejaVu Sans",
    "savefig.facecolor": "white",
})

print("Environment ready ✅")
print(f"Dataset expected at: {DATA_PATH.resolve()}")

## 2. Load the dataset and inspect its structure

This first pass fulfils the initial-inspection requirement: dataset shape, column names, data types, missing-value counts, and duplicate rows. We preserve `df_raw` so the original state can be compared with the cleaned version.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Download sales_data.csv from "
        "https://gist.github.com/summerofgeorge/33dbb09adbdea670c31ee2c4fc9f2617 "
        "and save it inside the project's data/ folder."
    )

df_raw = pd.read_csv(DATA_PATH)
print(f"Rows: {df_raw.shape[0]:,} | Columns: {df_raw.shape[1]:,}")
display(df_raw.head())
print("\nColumn data types:")
df_raw.info()
print("\nMissing values by column:")
display(df_raw.isna().sum().sort_values(ascending=False).rename("missing_count").to_frame())
print(f"\nExact duplicate rows: {df_raw.duplicated().sum():,}")

## 3. Standardise columns and clean the data

The file uses different naming styles in different versions of retail datasets. This cell standardises common aliases, converts identifiers to strings, parses dates, checks required columns, removes exact duplicate rows, and flags invalid dates or impossible numeric values. It does **not** silently invent missing age/gender/product data.

In [ ]:
# Convert column names like "Customer Age" to customer_age.
df = df_raw.copy()
df.columns = (
    df.columns.astype(str).str.strip().str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

# Common aliases: this keeps the notebook readable if the field names vary slightly.
aliases = {
    "date": "order_date",
    "transaction_date": "order_date",
    "customer_age": "customer_age",
    "age": "customer_age",
    "gender": "customer_gender",
    "customer_gender": "customer_gender",
    "product": "product_name",
    "item_purchased": "product_name",
    "item": "product_name",
    "product": "product_name",
    "product_category": "category",
    "product_name": "product_name",
    "quantity_sold": "quantity",
    "quantity": "quantity",
    "price": "unit_price",
    "price_per_unit": "unit_price",
    "unit_price": "unit_price",
}
rename_dict = {col: aliases[col] for col in df.columns if col in aliases and col != aliases[col]}
df = df.rename(columns=rename_dict)

required = ["order_date", "product_name", "quantity", "unit_price", "customer_age", "customer_gender"]
missing_required = [col for col in required if col not in df.columns]
if missing_required:
    raise ValueError(
        f"This dataset is missing required columns: {missing_required}. "
        "Choose a dataset with order date, product name, quantity, unit price, customer age and gender. "
        "Do not create fake product names or demographics just to fill the checklist."
    )

# Identifiers are labels, not measurements; keep them out of numeric summaries/correlations.
for col in ["order_id", "customer_id", "product_id"]:
    if col in df.columns:
        df[col] = df[col].astype("string")

# Parse and standardise.
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
for col in ["quantity", "unit_price", "customer_age"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["customer_gender", "category"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip().str.title()
df["product_name"] = (
    df["product_name"].astype("string").str.strip()
      .str.replace(r"\s+", " ", regex=True)
)

# Baseline quality metrics before the cleaning operations are finalised.
quality_before = {
    "rows": len(df),
    "duplicate_rows": int(df.duplicated().sum()),
    "missing_dates": int(df["order_date"].isna().sum()),
    "missing_product_names": int(df["product_name"].isna().sum()),
    "missing_quantity": int(df["quantity"].isna().sum()),
    "missing_unit_price": int(df["unit_price"].isna().sum()),
}

# Remove exact duplicates. Keep rows with missing demographics for sales totals, but mark their group as Unknown.
df = df.drop_duplicates().copy()
df["customer_gender"] = df["customer_gender"].fillna("Unknown").replace({"<NA>": "Unknown", "": "Unknown"})
if "category" not in df.columns:
    df["category"] = "Unspecified"
else:
    df["category"] = df["category"].fillna("Unspecified").replace({"<NA>": "Unspecified", "": "Unspecified"})

# Core transaction fields must be present. Keep a count of removed rows for transparent reporting.
rows_before_core_filter = len(df)
df = df.dropna(subset=["order_date", "product_name", "quantity", "unit_price"]).copy()
removed_core_rows = rows_before_core_filter - len(df)

# Drop impossible values only after exposing them for inspection.
invalid_quantity = int((df["quantity"] <= 0).sum())
invalid_price = int((df["unit_price"] < 0).sum())
invalid_age = int(((df["customer_age"] < 0) | (df["customer_age"] > 120)).sum())
print("Quality before cleaning:")
display(pd.Series(quality_before, name="value").to_frame())
print(f"Rows removed because a core transaction field was invalid/missing: {removed_core_rows:,}")
print(f"Rows with quantity <= 0 (inspect; may be returns): {invalid_quantity:,}")
print(f"Rows with unit price < 0 (inspect; may be refunds/adjustments): {invalid_price:,}")
print(f"Rows with age outside 0–120 (set to missing, not guessed): {invalid_age:,}")

# Treat implausible ages as missing rather than deleting the related sale.
df.loc[(df["customer_age"] < 0) | (df["customer_age"] > 120), "customer_age"] = np.nan
# Sales value is calculated consistently from quantity × unit price.
df["sales"] = df["quantity"] * df["unit_price"]

# Data quality summary after cleaning.
quality_after = pd.DataFrame({
    "before": pd.Series(quality_before),
    "after": pd.Series({
        "rows": len(df),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_dates": int(df["order_date"].isna().sum()),
        "missing_product_names": int(df["product_name"].isna().sum()),
        "missing_quantity": int(df["quantity"].isna().sum()),
        "missing_unit_price": int(df["unit_price"].isna().sum()),
    })
})
print("Before vs. after quality summary:")
display(quality_after)
print(f"Clean dataset ready: {len(df):,} rows")

## 4. Feature engineering and KPI snapshot

We add age groups, month labels, and quarter labels to support readable comparisons. `sales` is calculated as `quantity × unit_price`, because the selected source does not include a trustworthy transaction-total column. If negative quantities represent returns in your dataset, keep and document them rather than automatically removing them.

In [ ]:
# Use fixed, interpretable age bands. Missing ages remain Unknown.
age_bins = [-np.inf, 17, 24, 34, 44, 54, np.inf]
age_labels = ["Under 18", "18–24", "25–34", "35–44", "45–54", "55+"]
df["age_group"] = pd.cut(df["customer_age"], bins=age_bins, labels=age_labels, include_lowest=True)
df["age_group"] = df["age_group"].cat.add_categories(["Unknown"]).fillna("Unknown")
df["month_period"] = df["order_date"].dt.to_period("M")
df["month_label"] = df["order_date"].dt.strftime("%b %Y")
df["quarter_period"] = df["order_date"].dt.to_period("Q")

# Concise executive snapshot, calculated from the cleaned file.
summary_kpis = pd.Series({
    "Transactions / rows": len(df),
    "Unique customers": df["customer_id"].nunique() if "customer_id" in df.columns else np.nan,
    "Unique products": df["product_name"].nunique(),
    "Units sold (net)": df["quantity"].sum(),
    "Total sales value": df["sales"].sum(),
    "Average transaction-line value": df["sales"].mean(),
    "Date range": f"{df['order_date'].min():%d %b %Y} – {df['order_date'].max():%d %b %Y}",
})
display(summary_kpis.to_frame("value"))

## 5. Descriptive statistics for numerical measurement columns

Identifiers are stored as strings because their mean or standard deviation has no business meaning. The table below reports **mean, median, mode, and standard deviation** for the numeric measurements available in this dataset.

In [ ]:
numeric_cols = [c for c in ["customer_age", "quantity", "unit_price", "sales"] if c in df.columns]

def first_mode(series):
    modes = series.dropna().mode()
    return modes.iloc[0] if not modes.empty else np.nan

stats = pd.DataFrame({
    "mean": df[numeric_cols].mean(numeric_only=True),
    "median": df[numeric_cols].median(numeric_only=True),
    "mode": df[numeric_cols].apply(first_mode),
    "standard_deviation": df[numeric_cols].std(numeric_only=True),
    "minimum": df[numeric_cols].min(numeric_only=True),
    "maximum": df[numeric_cols].max(numeric_only=True),
}).round(2)
display(stats)
stats.to_csv(OUTPUT_DIR / "descriptive_statistics.csv")

## 6. Time-series analysis — monthly sales

The line chart uses the calendar month as the time unit. A peak identifies a period worth investigating; it does not by itself explain why sales rose or fell.

In [ ]:
monthly_sales = (
    df.set_index("order_date")["sales"]
      .resample("MS").sum()
      .rename("sales")
      .reset_index()
)
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.plot(monthly_sales["order_date"], monthly_sales["sales"], color=TEAL, linewidth=2.7, marker="o", markersize=6)
ax.fill_between(monthly_sales["order_date"], monthly_sales["sales"], color=TEAL, alpha=0.12)
ax.set_title("Monthly Sales Pulse", loc="left", fontsize=16, pad=14)
ax.set_xlabel("Month")
ax.set_ylabel("Sales value")
ax.tick_params(axis="x", rotation=35)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_monthly_sales.png", dpi=220, bbox_inches="tight")
plt.show()

peak_month = monthly_sales.loc[monthly_sales["sales"].idxmax()]
low_month = monthly_sales.loc[monthly_sales["sales"].idxmin()]
month_change = ((monthly_sales["sales"].iloc[-1] - monthly_sales["sales"].iloc[0]) / monthly_sales["sales"].iloc[0] * 100) if len(monthly_sales) > 1 and monthly_sales["sales"].iloc[0] != 0 else np.nan
display(Markdown(
    f"**Data-backed observation:** The highest-sales month was **{peak_month['order_date']:%B %Y}** "
    f"({peak_month['sales']:,.2f} sales value); the lowest was **{low_month['order_date']:%B %Y}** "
    f"({low_month['sales']:,.2f}). "
    + (f"Across the displayed range, the first-to-last monthly value changed by **{month_change:.1f}%**." if pd.notna(month_change) else "The first-to-last percentage change could not be calculated safely.")
))

### Markdown observation — monthly trend

Use the calculated observation above in your final report. Add one sentence on whether the monthly line is steady, volatile, or shows a clear spike, and mention the exact month. Avoid calling a single year's peak a recurring seasonal trend.

## 7. Time-series analysis — quarterly sales

Quarterly aggregation smooths some month-to-month noise and lets you compare broader periods.

In [ ]:
quarterly_sales = (
    df.set_index("order_date")["sales"]
      .resample("QS").sum()
      .rename("sales")
      .reset_index()
)
quarterly_sales["quarter"] = quarterly_sales["order_date"].dt.to_period("Q").astype(str)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(quarterly_sales["quarter"], quarterly_sales["sales"], color=AMBER, linewidth=2.7, marker="o", markersize=7)
ax.set_title("Quarterly Sales Trajectory", loc="left", fontsize=16, pad=14)
ax.set_xlabel("Quarter")
ax.set_ylabel("Sales value")
ax.spines[["top", "right"]].set_visible(False)
for x, y in enumerate(quarterly_sales["sales"]):
    ax.annotate(f"{y:,.0f}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9, color=INK)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "02_quarterly_sales.png", dpi=220, bbox_inches="tight")
plt.show()

if not quarterly_sales.empty:
    best_q = quarterly_sales.loc[quarterly_sales["sales"].idxmax()]
    worst_q = quarterly_sales.loc[quarterly_sales["sales"].idxmin()]
    display(Markdown(f"**Data-backed observation:** **{best_q['quarter']}** was the strongest quarter ({best_q['sales']:,.2f}), while **{worst_q['quarter']}** was the weakest ({worst_q['sales']:,.2f}). Compare this result with the monthly chart before suggesting a campaign calendar."))
quarterly_sales.to_csv(OUTPUT_DIR / "quarterly_sales.csv", index=False)
monthly_sales.to_csv(OUTPUT_DIR / "monthly_sales.csv", index=False)

### Markdown observation — quarterly trend

Record the best and weakest quarters from the generated note, then describe whether the quarterly pattern agrees with the monthly chart. If only one year of data is available, describe this as a pattern in the sample—not proof of a repeatable annual season.

## 8. Customer demographics — age-group distribution

In [ ]:
age_counts = df["age_group"].value_counts().reindex(age_labels + ["Unknown"], fill_value=0)
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.barplot(x=age_counts.index, y=age_counts.values, color=TEAL, ax=ax)
ax.set_title("Who Is Shopping? Transaction Count by Age Group", loc="left", fontsize=15, pad=12)
ax.set_xlabel("Customer age group")
ax.set_ylabel("Transaction rows")
ax.tick_params(axis="x", rotation=20)
ax.spines[["top", "right"]].set_visible(False)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x()+p.get_width()/2, p.get_height()), ha="center", va="bottom", fontsize=9, xytext=(0, 3), textcoords="offset points")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "03_age_group_distribution.png", dpi=220, bbox_inches="tight")
plt.show()

largest_age_group = age_counts.idxmax()
display(Markdown(f"**Data-backed observation:** The most represented age group by transaction count was **{largest_age_group}** ({int(age_counts.max()):,} rows). This chart counts transactions, not unique people or spending; use the revenue-by-group summary later before deciding which group is most valuable."))

### Markdown observation — age groups

State which age group contributes the most transaction rows and whether the distribution is balanced or concentrated. Mention that transaction counts are not the same as unique customers.

## 9. Customer demographics — gender breakdown

In [ ]:
gender_counts = df["customer_gender"].value_counts(dropna=False)
fig, ax = plt.subplots(figsize=(8, 5))
colors = [TEAL, AMBER, GREEN, CORAL, INK]
wedges, texts, autotexts = ax.pie(
    gender_counts.values,
    labels=gender_counts.index.astype(str),
    autopct=lambda pct: f"{pct:.1f}%",
    startangle=90,
    counterclock=False,
    colors=colors[:len(gender_counts)],
    wedgeprops={"linewidth": 2, "edgecolor": "white"},
    textprops={"color": INK, "fontsize": 10},
)
for t in autotexts:
    t.set_color("white")
    t.set_fontweight("bold")
ax.set_title("Customer Mix by Gender", loc="left", fontsize=15, pad=12)
ax.axis("equal")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "04_gender_breakdown.png", dpi=220, bbox_inches="tight")
plt.show()

lead_gender = gender_counts.idxmax()
display(Markdown(f"**Data-backed observation:** **{lead_gender}** is the largest gender group by transaction count ({int(gender_counts.max()):,} rows; {gender_counts.max()/gender_counts.sum()*100:.1f}% of rows). Use this as a descriptive sample pattern, not as evidence that gender caused higher spending."))

### Markdown observation — gender breakdown

Write the largest group and its approximate share. If the dataset has an `Unknown` group, note it and consider improving the source data before drawing customer-profile conclusions.

## 10. Product analysis — top 10 best-selling products

“Best-selling” is defined here as **total units sold**, not revenue. This distinction matters: a low-priced item can sell many units while a higher-priced product may generate more sales value.

In [ ]:
top_products_units = (
    df.groupby("product_name", dropna=False)["quantity"]
      .sum().sort_values(ascending=False).head(10).sort_values()
)
fig, ax = plt.subplots(figsize=(11, 6.5))
bar_colors = [AMBER if name == top_products_units.index[-1] else TEAL for name in top_products_units.index]
ax.barh(top_products_units.index.astype(str), top_products_units.values, color=bar_colors, edgecolor="white")
ax.set_title("Top 10 Products by Units Sold", loc="left", fontsize=16, pad=14)
ax.set_xlabel("Units sold")
ax.set_ylabel("Product")
ax.spines[["top", "right"]].set_visible(False)
for y, value in enumerate(top_products_units.values):
    ax.text(value, y, f"  {value:,.0f}", va="center", fontsize=9, color=INK)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "05_top_10_products.png", dpi=220, bbox_inches="tight")
plt.show()

best_seller = top_products_units.index[-1]
best_units = top_products_units.iloc[-1]
all_units = df["quantity"].sum()
best_units_share = (best_units / all_units * 100) if all_units else np.nan
display(Markdown(f"**Data-backed observation:** **{best_seller}** had the highest net unit quantity in this sample ({best_units:,.0f} units; {best_units_share:.1f}% of all units if the total is non-zero). This is a unit-volume ranking, not a profit ranking."))
top_products_units.sort_values(ascending=False).rename("units_sold").to_csv(OUTPUT_DIR / "top_10_products_by_units.csv")

### Markdown observation — top products

Name the product with the highest unit sales. Compare the height of the top bar with the rest: is demand concentrated in one item, or spread across multiple products? Only make inventory recommendations after checking the revenue chart and whether returns/negative quantities are present.

## 11. Product analysis — revenue by category

In [ ]:
category_revenue = (
    df.groupby("category", dropna=False)["sales"]
      .sum().sort_values(ascending=False)
)
fig, ax = plt.subplots(figsize=(10, 5.5))
bar_palette = [AMBER if name == category_revenue.idxmax() else TEAL for name in category_revenue.index]
ax.bar(category_revenue.index.astype(str), category_revenue.values, color=bar_palette, edgecolor="white")
ax.set_title("Where Does the Sales Value Come From?", loc="left", fontsize=16, pad=14)
ax.set_xlabel("Product category")
ax.set_ylabel("Sales value")
ax.tick_params(axis="x", rotation=15)
ax.spines[["top", "right"]].set_visible(False)
for p in ax.patches:
    ax.annotate(f"{p.get_height():,.0f}", (p.get_x()+p.get_width()/2, p.get_height()), ha="center", va="bottom", fontsize=9, xytext=(0, 3), textcoords="offset points")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "06_category_revenue.png", dpi=220, bbox_inches="tight")
plt.show()

best_category = category_revenue.idxmax()
best_category_sales = category_revenue.max()
category_share = best_category_sales / category_revenue.sum() * 100 if category_revenue.sum() else np.nan
display(Markdown(f"**Data-backed observation:** **{best_category}** generated the highest category-level sales value ({best_category_sales:,.2f}; {category_share:.1f}% of total calculated sales if total sales are non-zero). Revenue alone does not reveal profit or margin."))
category_revenue.rename("sales").to_csv(OUTPUT_DIR / "revenue_by_category.csv")

### Markdown observation — category revenue

Record the leading category and its share of total calculated sales. Do not call it the most profitable category: the source has no reliable cost-of-goods data, so this chart measures sales value only.

## 12. Correlation heatmap — numerical variables

Correlation measures linear association between two numeric variables. It cannot establish causation. Because `sales` is calculated from `quantity × unit_price`, its correlation with those two variables is partly built into the formula; interpret those cells accordingly.

In [ ]:
corr_cols = [c for c in ["customer_age", "quantity", "unit_price", "sales"] if c in df.columns]
corr = df[corr_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(8.5, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="YlGnBu", center=0, vmin=-1, vmax=1,
            linewidths=0.8, square=True, cbar_kws={"shrink": 0.82}, ax=ax)
ax.set_title("Numerical Relationship Map", loc="left", fontsize=16, pad=14)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "07_correlation_heatmap.png", dpi=220, bbox_inches="tight")
plt.show()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
if not upper.empty:
    strongest_pair = upper.abs().idxmax()
    strongest_value = upper.loc[strongest_pair]
    display(Markdown(f"**Data-backed observation:** The strongest absolute pairwise correlation in the displayed matrix was between **{strongest_pair[0]}** and **{strongest_pair[1]}** (r = {strongest_value:.2f}). Check whether the relationship is mathematically expected before interpreting it as a business discovery."))

### Markdown observation — correlation

Identify the strongest visible relationship and its direction (positive or negative). Then state whether it is business-relevant or partly expected because sales were calculated from quantity multiplied by unit price. Do not use the word “causes.”

## 13. Extra insight — revenue concentration / Pareto view

This is the additional visualisation beyond the required checklist. The bars show the highest-revenue products; the line shows their cumulative contribution to **all** product revenue. It helps answer whether a small number of products account for a large share of sales.

In [ ]:
product_revenue = (
    df.groupby("product_name")["sales"]
      .sum().sort_values(ascending=False)
)
top10_revenue = product_revenue.head(10)
cumulative_share = top10_revenue.cumsum() / product_revenue.sum() * 100 if product_revenue.sum() else top10_revenue.cumsum() * np.nan

fig, ax1 = plt.subplots(figsize=(12, 6.5))
positions = np.arange(len(top10_revenue))
ax1.bar(positions, top10_revenue.values, color=TEAL, edgecolor="white", label="Sales value")
ax1.set_xticks(positions)
ax1.set_xticklabels(top10_revenue.index.astype(str), rotation=35, ha="right")
ax1.set_ylabel("Sales value", color=INK)
ax1.set_xlabel("Product")
ax1.set_title("Revenue Concentration: Do a Few Products Carry the Sales?", loc="left", fontsize=15, pad=14)
ax1.spines[["top", "right"]].set_visible(False)

ax2 = ax1.twinx()
ax2.plot(positions, cumulative_share.values, color=AMBER, marker="o", linewidth=2.5, label="Cumulative share of all revenue")
ax2.set_ylabel("Cumulative share of total sales (%)", color=INK)
ax2.set_ylim(0, 105)
ax2.axhline(80, color=CORAL, linestyle="--", linewidth=1.2, alpha=0.8)
ax2.text(max(len(positions)-1, 0), 81.5, "80% reference", ha="right", color=CORAL, fontsize=9)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "08_product_revenue_pareto.png", dpi=220, bbox_inches="tight")
plt.show()

if len(top10_revenue) >= 3 and product_revenue.sum() != 0:
    top3_share = product_revenue.head(3).sum() / product_revenue.sum() * 100
    display(Markdown(f"**Non-obvious insight:** The top three products contributed **{top3_share:.1f}%** of total calculated sales value. " + ("That indicates comparatively concentrated product revenue; protect availability of these items while testing alternatives." if top3_share >= 50 else "Revenue is not dominated by the top three products alone in this sample; avoid over-investing in a single item based only on rank.")))
else:
    top3_share = np.nan
    display(Markdown("**Non-obvious insight:** There is not enough valid product revenue to calculate a meaningful top-three concentration share."))

### Markdown observation — Pareto / concentration

Record the top-three revenue share shown above. Explain whether the business seems dependent on a small group of items. This is different from the units-sold ranking: a product can sell fewer units but still contribute a large sales value.

## 14. Compare the customer groups by sales value

This additional table helps avoid mistaking a large group for a high-value group. It is also a useful cross-check for the age and gender charts.

In [ ]:
age_revenue = df.groupby("age_group", observed=False)["sales"].agg(["sum", "mean", "count"]).sort_values("sum", ascending=False)
gender_revenue = df.groupby("customer_gender")["sales"].agg(["sum", "mean", "count"]).sort_values("sum", ascending=False)
print("Sales value by age group:")
display(age_revenue.round(2))
print("Sales value by gender group:")
display(gender_revenue.round(2))
age_revenue.to_csv(OUTPUT_DIR / "sales_by_age_group.csv")
gender_revenue.to_csv(OUTPUT_DIR / "sales_by_gender.csv")

## 15. Three evidence-based business recommendations

The recommendations below are generated from the actual values in the dataset. They are hypotheses to test, not guaranteed outcomes. Since this starter dataset is simulated, validate these recommendations with real retailer records before using them operationally.

In [ ]:
# Find values safely so the cell continues to work with small datasets.
valid_age_revenue = age_revenue.loc[age_revenue.index != "Unknown"].dropna(subset=["sum"])
if not valid_age_revenue.empty:
    leading_age = valid_age_revenue["sum"].idxmax()
    leading_age_sales = valid_age_revenue.loc[leading_age, "sum"]
    leading_age_share = leading_age_sales / df["sales"].sum() * 100 if df["sales"].sum() else np.nan
else:
    leading_age, leading_age_sales, leading_age_share = "Unknown", np.nan, np.nan

valid_gender_revenue = gender_revenue.dropna(subset=["sum"])
if not valid_gender_revenue.empty:
    leading_gender_value = valid_gender_revenue["sum"].idxmax()
    leading_gender_sales = valid_gender_revenue.loc[leading_gender_value, "sum"]
else:
    leading_gender_value, leading_gender_sales = "Unknown", np.nan

valid_months = monthly_sales[monthly_sales["sales"].notna()]
if not valid_months.empty:
    peak_month_name = valid_months.loc[valid_months["sales"].idxmax(), "order_date"].strftime("%B %Y")
    peak_month_value = valid_months["sales"].max()
else:
    peak_month_name, peak_month_value = "unavailable", np.nan

recommendations = [
    f"**Inventory test:** Prioritise availability checks for **{best_seller}**, the highest-unit product in this sample ({best_units:,.0f} units). Compare stockout and margin records before increasing purchase quantities.",
    f"**Category test:** Review pricing, stock depth, and campaign placement for **{best_category}**, which leads calculated category sales ({category_share:.1f}% of sales value). Test one targeted promotion and compare conversion and margin against a control period.",
    f"**Campaign-calendar test:** The strongest month in this sample was **{peak_month_name}** ({peak_month_value:,.2f} sales value). Plan a small campaign or stock-readiness check before that period in a future comparable period, then verify whether the pattern repeats.",
]
if pd.notna(leading_age_share):
    recommendations.append(f"**Customer message test:** The **{leading_age}** age group accounted for {leading_age_share:.1f}% of calculated sales value among all rows. A/B test product messaging for this segment while checking sample size and customer experience; do not assume age itself causes purchasing behaviour.")

for i, recommendation in enumerate(recommendations[:3], start=1):
    display(Markdown(f"{i}. {recommendation}"))

## 16. Export deliverables for your internship submission

The notebook, cleaned CSV, summary tables, and chart images make the project easier to review. Keep the raw dataset unchanged in `data/`, and submit the notebook with outputs visible after running all cells.

In [ ]:
# Export cleaned data and a one-row KPI summary.
clean_export = df.copy()
clean_export["month_period"] = clean_export["month_period"].astype(str)
clean_export["quarter_period"] = clean_export["quarter_period"].astype(str)
clean_export.to_csv(OUTPUT_DIR / "retail_sales_cleaned.csv", index=False)

kpi_export = pd.DataFrame([{
    "rows_after_cleaning": len(df),
    "unique_products": df["product_name"].nunique(),
    "total_units": df["quantity"].sum(),
    "total_sales_value": df["sales"].sum(),
    "average_line_sales": df["sales"].mean(),
    "top_unit_product": best_seller,
    "top_unit_product_quantity": best_units,
    "top_category_by_sales": best_category,
    "top_category_sales_share_pct": category_share,
    "top_3_products_sales_share_pct": top3_share,
    "peak_month": peak_month_name,
    "peak_month_sales": peak_month_value,
}])
kpi_export.to_csv(OUTPUT_DIR / "executive_kpis.csv", index=False)

print("Export complete ✅")
print(f"Cleaned dataset: {OUTPUT_DIR / 'retail_sales_cleaned.csv'}")
print(f"Executive KPIs:  {OUTPUT_DIR / 'executive_kpis.csv'}")
print(f"Charts folder:   {FIGURE_DIR}")
print("Before submission: Kernel → Restart Kernel and Run All Cells, then save the notebook with outputs visible.")

## 17. Final conclusion — complete this after you review the outputs

Use the automatically generated metrics and charts to write a short executive conclusion in your own words. It should contain:

1. **Sales trend:** name the strongest month and quarter, and describe the overall pattern.
2. **Customer behaviour:** identify the leading age and gender groups by transaction count, then compare their sales-value tables.
3. **Product performance:** name the top product by units and leading category by sales value; compare this with the Pareto concentration result.
4. **At least three actionable recommendations:** take the generated recommendations above, validate them against the charts, and rewrite them in your own words.
5. **Limitations:** state that the supplied source is simulated, the analysis is descriptive, sales value is not profit, and one year of observations is not enough to prove recurring seasonality.

### Suggested final paragraph format

> The analysis of **[number of rows]** transaction rows across **[date range]** indicates that **[peak period]** was the strongest observed period. **[top product]** led unit volume, while **[top category]** generated the highest category sales value. The product concentration chart shows that the top three products account for **[share]%** of calculated sales. Based on these patterns, the business should test **[recommendation 1]**, **[recommendation 2]**, and **[recommendation 3]**. These findings are descriptive and should be validated using real sales, cost, inventory, and customer records before operational decisions are made.